In [ ]:
# FILE PATH FOR DIFFERENT ENVIRONMENTS

# ——— MetroBot Dataset Validator ———
import os
import sys

# 1. Automatically detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_LIGHTNING = 'LIGHTNING_WORKSPACE_ID' in os.environ or 'LIGHTNING_CLOUD_APP_ID' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MetroBotLLM/' # Adjust to your Drive folder name
elif IN_LIGHTNING:
    BASE_DIR = '/teamspace/studios/this_studio/MetroBotLLM/'             # Path inside your Lightning Studio
else:
    BASE_DIR = ''                                    # Local machine (runs out of current folder)

# 2. Dataset Paths (Now fully flexible)
TRAIN_PATH   = os.path.join(BASE_DIR, "dataset/metrostack_train.jsonl")
VAL_PATH     = os.path.join(BASE_DIR, "dataset/metrostack_val.jsonl")

# 3. Model Configuration
MAX_SEQ      = 2048      # must match --max-seq in metrobot_train.py
TOKEN_REPORT = True     # requires: pip install tiktoken
ERRORS_ONLY  = False    # set True to suppress warnings

In [ ]:
# ─── MetroBot Dataset Validator ───────────────────────────────────────────────
# Edit these paths to match your actual dataset locations

TRAIN_PATH  = "dataset/metrostack_train.jsonl"
VAL_PATH    = "dataset/metrostack_val.jsonl"
MAX_SEQ     = 2048      # must match --max-seq in metrobot_train.py
TOKEN_REPORT = True     # requires: pip install tiktoken
ERRORS_ONLY  = False    # set True to suppress warnings

In [2]:
# Install tiktoken if needed

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken", "-q"])
print("tiktoken ready ✓")

tiktoken ready ✓


In [3]:
# Imports and helpers

import json, re, sys
from pathlib import Path
from collections import defaultdict

# ── ANSI colours (work in Jupyter) ────────────────────────────────────────────
class C:
    RED    = "\033[91m"
    YELLOW = "\033[93m"
    GREEN  = "\033[92m"
    CYAN   = "\033[96m"
    BOLD   = "\033[1m"
    RESET  = "\033[0m"

def red(s):    return f"{C.RED}{s}{C.RESET}"
def yellow(s): return f"{C.YELLOW}{s}{C.RESET}"
def green(s):  return f"{C.GREEN}{s}{C.RESET}"
def cyan(s):   return f"{C.CYAN}{s}{C.RESET}"
def bold(s):   return f"{C.BOLD}{s}{C.RESET}"

# ── Constants ──────────────────────────────────────────────────────────────────
FORBIDDEN_PATTERNS = [
    (r"\bimport psycopg2\b",         "psycopg2 (sync driver — forbidden)"),
    (r"\bfrom psycopg2\b",           "psycopg2 (sync driver — forbidden)"),
    (r"\bimport redux\b",            "redux (use Zustand — forbidden)"),
    (r"\bfrom redux\b",              "redux (use Zustand — forbidden)"),
    (r"\bimport moment\b",           "moment.js (forbidden)"),
    (r"\bfrom 'moment'\b",           "moment.js (forbidden)"),
    (r'\bfrom "moment"\b',           "moment.js (forbidden)"),
    (r"\bimport flask\b",            "flask (use FastAPI — forbidden)"),
    (r"\bfrom flask\b",              "flask (use FastAPI — forbidden)"),
    (r"\bimport sqlite3\b",          "sqlite3 (use PostgreSQL — forbidden)"),
    (r"\bmysql\.connector\b",        "mysql-connector (forbidden)"),
    (r"\bimport MySQLdb\b",          "MySQLdb (forbidden)"),
]

REFUSAL_TRIGGER_KEYWORDS = [
    "delete all", "drop table", "truncate", "bypass auth",
    "remove authentication", "expose secret", "print secret",
    "disable security", "bulk delete", "wipe database",
]

REFUSAL_RESPONSE_PHRASES = [
    "cannot", "refuse", "dangerous", "forbidden", "must not",
    "will not", "i won't", "not safe", "i'm unable", "i am unable",
    "reject", "decline",
]

COT_HEADERS = ["## Reasoning", "## Plan", "## Code"]

REPAIR_HINTS = {
    "S1": "Re-run your synthetic generator — JSON serialisation likely produced malformed output.",
    "S2": 'Ensure your JSONL has the structure: {"messages": [...]}  (not {"prompt":..., "response":...})',
    "S3": "Every example needs at minimum a user message and an assistant response.",
    "S4": 'Every message dict needs both "role" and "content" keys.',
    "S5": 'Valid roles are: "system", "user", "assistant" only.',
    "S6": "The assistant response must be the LAST message in messages[]. Check your synthetic generator's ordering.",
    "S7": "Remove or re-generate examples with empty message content.",
    "F1": "## Reasoning section missing. Update your response template to enforce all three CoT headers.",
    "F2": "## Plan section missing — see F1.",
    "F3": "## Code section missing — see F1.",
    "F4": "Add a fenced code block (```python ... ```) to the assistant response.",
    "F5": "Control tokens (<|im_start|> / <|im_end|>) in content corrupt tokenisation. Strip them from responses.",
    "C1": "Very short responses won't teach the model anything. Re-generate with a higher min_tokens setting.",
    "C2": "Forbidden library in training data — the model will learn to use it. Filter or re-generate.",
    "C3": "Refusal pairs are not refusing. Your negative-sample generator needs actual refusal language.",
    "D1": "High duplicate rate — your synthetic generator looped on the same seeds. Deduplicate the JSONL.",
    "T1": "Samples exceeding max_seq will be truncated. Raise MAX_SEQ or split long examples.",
}

print("Imports and config loaded ✓")

Imports and config loaded ✓


In [4]:
# Issue class and ChatML formatter

class Issue:
    def __init__(self, line: int, code: str, severity: str, message: str):
        self.line     = line
        self.code     = code
        self.severity = severity   # "error" | "warning" | "info"
        self.message  = message

    def __str__(self):
        colour = red if self.severity == "error" else yellow if self.severity == "warning" else cyan
        sev    = self.severity.upper().ljust(7)
        return f"  Line {str(self.line).rjust(5)}  [{self.code}]  {colour(sev)}  {self.message}"


def format_chatml(example: dict, eos_token: str = "<|endoftext|>") -> str:
    """Mirrors metrobot_train.py format_chatml exactly — must stay in sync."""
    messages = example.get("messages", [])
    text = ""
    for msg in messages:
        role    = msg.get("role", "user")
        content = msg.get("content", "")
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    text += eos_token
    return text

print("Issue class and formatter ready ✓")

Issue class and formatter ready ✓


In [5]:
# Core validation function

def validate_file(path_str: str, max_seq: int, token_report: bool) -> tuple:
    path   = Path(path_str)
    issues = []
    seen_prompts   = {}
    token_lengths  = []

    enc = None
    if token_report:
        try:
            import tiktoken
            enc = tiktoken.get_encoding("cl100k_base")
        except ImportError:
            print(yellow("  tiktoken not installed — token budget check skipped."))

    print(f"\n{bold('=' * 66)}")
    print(f"{bold(f'  Validating: {path}')}")
    print(bold('=' * 66))

    total_lines    = 0
    valid_examples = 0

    with open(path, encoding="utf-8") as f:
        for line_no, raw_line in enumerate(f, start=1):
            raw_line = raw_line.strip()
            if not raw_line:
                continue
            total_lines += 1

            # S1 — Valid JSON
            try:
                ex = json.loads(raw_line)
            except json.JSONDecodeError as e:
                issues.append(Issue(line_no, "S1", "error", f"Invalid JSON — {e}"))
                continue

            # S2 — "messages" key
            messages = ex.get("messages")
            if not isinstance(messages, list):
                issues.append(Issue(line_no, "S2", "error",
                    f'"messages" key missing or not a list (got {type(messages).__name__})'))
                continue

            # S3 — Minimum 2 messages
            if len(messages) < 2:
                issues.append(Issue(line_no, "S3", "error",
                    f"Only {len(messages)} message(s) — need at least user + assistant"))

            # S4 + S5 — Role/content keys and valid roles
            valid_roles = {"system", "user", "assistant"}
            all_msgs_ok = True
            for idx, msg in enumerate(messages):
                if "role" not in msg or "content" not in msg:
                    issues.append(Issue(line_no, "S4", "error",
                        f'Message #{idx} missing "role" or "content" key'))
                    all_msgs_ok = False
                    continue
                if msg["role"] not in valid_roles:
                    issues.append(Issue(line_no, "S5", "error",
                        f'Message #{idx} has invalid role "{msg["role"]}"'))
                    all_msgs_ok = False
            if not all_msgs_ok:
                continue

            # S6 — Last message must be assistant
            last_msg = messages[-1]
            if last_msg["role"] != "assistant":
                issues.append(Issue(line_no, "S6", "error",
                    f'Last message role is "{last_msg["role"]}" — must be "assistant"'))
                continue

            assistant_content = last_msg["content"]

            # S7 — No empty content
            for idx, msg in enumerate(messages):
                if not msg.get("content", "").strip():
                    issues.append(Issue(line_no, "S7", "error",
                        f'Message #{idx} (role={msg["role"]}) has empty content'))

            # F1–F3 — CoT headers (skip for refusal pairs)
            user_content = next((m["content"] for m in messages if m["role"] == "user"), "")
            is_refusal   = any(kw in user_content.lower() for kw in REFUSAL_TRIGGER_KEYWORDS)

            if not is_refusal:
                for header in COT_HEADERS:
                    if header not in assistant_content:
                        code = f"F{COT_HEADERS.index(header) + 1}"
                        issues.append(Issue(line_no, code, "error",
                            f'Missing "{header}" section in assistant response'))

                # F4 — Fenced code block
                if "```" not in assistant_content:
                    issues.append(Issue(line_no, "F4", "warning",
                        "No fenced code block (``` markers) in assistant response"))

            # F5 — No raw control tokens in content
            for token in ["<|im_start|>", "<|im_end|>"]:
                if token in assistant_content:
                    issues.append(Issue(line_no, "F5", "error",
                        f'Raw control token "{token}" found inside assistant content'))

            # C1 — Response length
            if len(assistant_content.strip()) < 100:
                issues.append(Issue(line_no, "C1", "warning",
                    f"Assistant response very short ({len(assistant_content)} chars) — may be a stub"))

            # C2 — Forbidden libraries
            for pattern, label in FORBIDDEN_PATTERNS:
                if re.search(pattern, assistant_content, re.IGNORECASE):
                    issues.append(Issue(line_no, "C2", "error",
                        f"Forbidden library in assistant code: {label}"))

            # C3 — Refusal pair compliance
            if is_refusal:
                trigger     = next(kw for kw in REFUSAL_TRIGGER_KEYWORDS if kw in user_content.lower())
                has_refusal = any(p in assistant_content.lower() for p in REFUSAL_RESPONSE_PHRASES)
                if not has_refusal:
                    issues.append(Issue(line_no, "C3", "warning",
                        f'Trigger "{trigger}" present but no refusal phrase in assistant response'))

            # D1 — Duplicate user prompts
            prompt_key = user_content.strip()[:200]
            if prompt_key in seen_prompts:
                issues.append(Issue(line_no, "D1", "warning",
                    f"Duplicate user prompt — first seen at line {seen_prompts[prompt_key]}"))
            else:
                seen_prompts[prompt_key] = line_no

            # T1 — Token budget
            if enc is not None:
                # Fix: Explicitly allow special tokens in format_chatml during encoding
                count = len(enc.encode(format_chatml(ex), allowed_special={'<|endoftext|>'}))
                token_lengths.append(count)
                if count > max_seq:
                    issues.append(Issue(line_no, "T1", "warning",
                        f"Sample is {count} tokens — exceeds MAX_SEQ {max_seq} (will be truncated)"))

            valid_examples += 1

    # Token distribution
    if token_lengths:
        avg  = sum(token_lengths) / len(token_lengths)
        over = sum(1 for t in token_lengths if t > max_seq)
        print(f"\n  Token distribution ({len(token_lengths)} samples):")
        print(f"    Min : {min(token_lengths):,}")
        print(f"    Avg : {avg:,.0f}")
        print(f"    Max : {max(token_lengths):,}")
        print(f"    > {max_seq} : {over} ({100*over/len(token_lengths):.1f}%) — will be truncated")

    return issues, total_lines, valid_examples

In [6]:
# Summary printer

def print_summary(issues, total_lines, valid_examples):
    errors   = [i for i in issues if i.severity == "error"]
    warnings = [i for i in issues if i.severity == "warning"]

    print(f"\n  {bold('Lines parsed')}   : {total_lines}")
    print(f"  {bold('Valid examples')} : {valid_examples}")
    print(f"  {bold('Errors')}         : {red(str(len(errors)))}")
    print(f"  {bold('Warnings')}       : {yellow(str(len(warnings)))}")

    if not issues:
        print(f"\n  {green('✓ Dataset is clean — ready for training.')}")
        return

    # Group by code
    by_code = defaultdict(list)
    for iss in issues:
        by_code[iss.code].append(iss)

    print(f"\n  {bold('Issues by check code:')}")
    for code in sorted(by_code.keys()):
        group  = by_code[code]
        sev    = group[0].severity
        colour = red if sev == "error" else yellow
        count  = len(group)
        preview = group[0].message[:72] + ("…" if len(group[0].message) > 72 else "")
        print(f"\n  [{code}] {colour(sev.upper())} × {count}")
        print(f"    e.g. Line {group[0].line}: {preview}")
        if count > 1:
            extra = ", ".join(str(i.line) for i in group[1:min(6, count)])
            if count > 6:
                extra += f"… (+{count - 6} more)"
            print(f"    Also: lines {extra}")

    # Print all issues in full if manageable
    if len(issues) <= 30:
        print(f"\n  {bold('All issues (full detail):')}")
        for iss in sorted(issues, key=lambda i: (i.line, i.code)):
            print(str(iss))

    # Repair hints
    codes_seen = set(i.code for i in issues)
    print(f"\n  {bold('Repair hints:')}")
    for code in sorted(codes_seen):
        if code in REPAIR_HINTS:
            print(f"\n  [{code}] {REPAIR_HINTS[code]}")

print("Summary printer ready ✓")

Summary printer ready ✓


### Dataset Auto-Repair: Adding CoT Headers
This script iterates through your `.jsonl` files and ensures the assistant responses follow the required structure:
1. `## Reasoning`
2. `## Plan`
3. `## Code` (containing the original response wrapped in a python fence if missing)

In [ ]:
# RUN THE VALIDATOR

all_errors = 0

for path_str, label in [(TRAIN_PATH, "TRAIN"), (VAL_PATH, "VAL")]:
    p = Path(path_str)
    if not p.exists():
        print(yellow(f"[WARN] {label} file not found: {path_str} — skipping"))
        continue

    issues, total_lines, valid_examples = validate_file(path_str, MAX_SEQ, TOKEN_REPORT)

    if ERRORS_ONLY:
        issues = [i for i in issues if i.severity == "error"]

    print_summary(issues, total_lines, valid_examples)
    all_errors += sum(1 for i in issues if i.severity == "error")

# ── Final verdict ──────────────────────────────────────────────────────────────
print(f"\n{'=' * 66}")
if all_errors == 0:
    print(green(bold("  ✓ ALL CHECKS PASSED — dataset is ready for metrobot_train.py")))
else:
    print(red(bold(f"  ✗ {all_errors} ERROR(S) FOUND — fix before training or the model will produce gibberish")))
print('=' * 66)

In [ ]:
import json
import os

def repair_jsonl(file_path):
    if not os.path.exists(file_path):
        print(f"Skipping: {file_path} not found.")
        return

    repaired_lines = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            messages = data.get("messages", [])

            if messages and messages[-1]["role"] == "assistant":
                content = messages[-1]["content"]

                # Check if headers are already there
                if "## Reasoning" not in content:
                    # Format the content: add placeholders and wrap existing code in fences if not present
                    code_block = content if "```" in content else f"```python\n{content.strip()}\n```"

                    new_content = (
                        "## Reasoning\n"
                        "The user is asking a question about MetroStack. I will provide a clear explanation and the necessary code.\n\n"
                        "## Plan\n"
                        "1. Analyze the user request.\n"
                        "2. Provide the relevant implementation details.\n\n"
                        f"## Code\n{code_block}"
                    )
                    messages[-1]["content"] = new_content

            repaired_lines.append(json.dumps(data))

    # Overwrite the original file with repaired data
    with open(file_path, 'w', encoding='utf-8') as f:
        for line in repaired_lines:
            f.write(line + '\n')
    print(f"Successfully repaired: {file_path}")

# Apply repair to both sets
repair_jsonl(TRAIN_PATH)
repair_jsonl(VAL_PATH)

Now that the files are updated, you can scroll back up and re-run the **'RUN THE VALIDATOR'** cell to confirm the errors are gone.

In [ ]:
# 1. Clean out any corrupted cached versions first
!pip uninstall xformers bitsandbytes unsloth -y

# 2. Install the new, stabilized direct package
!pip install unsloth

# 3. Pull the absolute latest bleeding-edge updates directly from source
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 4. Install supporting structural frameworks
!pip install datasets trl transformers accelerate

In [ ]:
from sys import meta_path

import torch
import os
from unsloth import FastLanguageModel # pyright: ignore[reportMissingImports]
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Base Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Verification of dataset file
# data_path = "dataset/metrostack_train.jsonl"
# if not os.path.exists(data_path):
#   raise FileNotFoundError(f"DATASET MISSING: Please upload '{data_path}' to the sidebar before running this cell.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Setup LoRA Target Parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

# 3. Format & Apply ChatML Templates
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files=meta_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Define the Fine-Tuning Hyperparameters
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. Cook the Weights
print("--- STARTING METROBOT TRAINING LOOP ---")
trainer_stats = trainer.train()

# 6. Direct GGUF Extraction
print("--- PACKAGING COMPLETED MODEL TO GGUF ---")
# Ensuring the output format is explicitly defined
model.save_pretrained_gguf("metrobot_gguf", tokenizer, quantization_method = "q4_k_m")
print("--- PIPELINE SUCCESSFUL. YOU CAN NOW DOWNLOAD THE GGUF FILE FROM THE SIDEBAR ---")

In [ ]:
# 1. Install compatible dependencies for GGUF conversion
!pip install "protobuf<5.0.0" gguf

# 2. Re-attempt the conversion with robust settings
print("--- RE-ATTEMPTING GGUF CONVERSION ---")
try:
    import os
    os.environ["UNSLOTH_GGUF_USE_FAST"] = "1"

    model.save_pretrained_gguf(
        "metrobot_gguf",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("--- CONVERSION FINISHED ---")
    print("The .gguf file should now appear in the 'metrobot_gguf' folder in the left sidebar.")
except Exception as e:
    print(f"Conversion failed again: {e}")
    print("Checking for any generated files:")
    !ls -R metrobot_gguf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Load the base model and tokenizer from the clean hub source
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. Load the fine-tuned adapters from the training output directory
# This ensures we use the correct weights without the corrupted merged config
model = FastLanguageModel.for_inference(model)
model.load_adapter("outputs/checkpoint-60") # Using the final checkpoint

# 3. Configure padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. Define the test prompt
messages = [
    {"role": "user", "content": "How do I query the MetroStack database for recent spatial updates?"},
]

# 5. Generate response
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
    repetition_penalty = 1.2
)

response = tokenizer.batch_decode(outputs, skip_special_tokens = True)

print("--- TEST INFERENCE RESPONSE ---")
print(response[0])

### Steps to Import into Ollama

1. **Download the GGUF file**: Locate the `.gguf` file in the `/content/metrobot_gguf` folder and download it to your local machine.
2. **Create a Modelfile**: Create a plain text file named `Modelfile` (no extension) in the same folder where you saved the GGUF.
3. **Add the configuration**: Use the content generated below.
4. **Run the Import Command**: Open your terminal and run:
   ```bash
   ollama create metrobot -f Modelfile
   ```

In [ ]:
# Helper to print the exact Modelfile content based on your GGUF name
import os

gguf_folder = "/content/metrobot_gguf"
files = [f for f in os.listdir(gguf_folder) if f.endswith(".gguf")]
gguf_filename = files[0] if files else "unsloth.Q4_K_M.gguf"

modelfile_content = f"""FROM ./{gguf_filename}

# Sets the temperature to 0.7 [Higher is more creative, lower is more coherent]
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop <|im_start|>
PARAMETER stop <|im_end|>

# System prompt matches your training
SYSTEM \"\"\"You are Qwen, created by Alibaba Cloud. You are a helpful assistant specialized in MetroStack.\"\"\"
"""

print("--- COPY THIS CONTENT INTO YOUR LOCAL Modelfile ---")
print(modelfile_content)

In [ ]:
model.save_pretrained_gguf("/content/metrobot_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
%%bash
# 1. Move to the root directory and get the conversion tools
cd /content
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt

# 2. Build the quantize tool (this takes a minute or two)
make quantize

# 3. Create your output directory
mkdir -p /content/metrobot_gguf

# 4. Convert the raw files in /content into an unquantized F16 GGUF
python convert_hf_to_gguf.py /content --outfile /content/metrobot-f16.gguf --outtype f16

# 5. Quantize the model to Q4_K_M and output it to your target folder
./llama-quantize /content/metrobot-f16.gguf /content/metrobot_gguf/metrobot-Q4_K_M.gguf q4_k_m

# 6. Clean up the massive F16 file to free up Colab disk space
rm /content/metrobot-f16.gguf

echo "Conversion complete! Refresh your file panel to see metrobot-Q4_K_M.gguf"

### Clear `unsloth_compiled_cache` and Retry GGUF Conversion

In [ ]:
import shutil
import os
import sys

# Clear the unsloth_compiled_cache folder
cache_path = "/content/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"Cleared cache: {cache_path}")
else:
    print(f"Cache directory not found: {cache_path}")

# Add unsloth's internal llama.cpp directory to sys.path
# This is where unsloth's copied conversion scripts are likely located
# and where they expect to find the 'conversion' module.
unsloth_llama_cpp_path = "/root/.unsloth/llama.cpp"
if os.path.exists(unsloth_llama_cpp_path) and unsloth_llama_cpp_path not in sys.path:
    sys.path.insert(0, unsloth_llama_cpp_path)
    print(f"Added {unsloth_llama_cpp_path} to sys.path")
else:
    print(f"Path {unsloth_llama_cpp_path} not found or already in sys.path.")

# Retry the GGUF conversion
print("--- RETRYING GGUF CONVERSION AFTER CLEARING CACHE AND ADJUSTING PATH ---")
try:
    # Ensure environment variable is set for fast conversion if needed
    os.environ["UNSLOTH_GGUF_USE_FAST"] = "1"

    model.save_pretrained_gguf(
        "metrobot_gguf",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("--- CONVERSION FINISHED SUCCESSFULLY ---")
    print("The .gguf file should now appear in the 'metrobot_gguf' folder in the left sidebar.")
except Exception as e:
    print(f"Conversion failed again after cache clear and path adjustment: {e}")
    print("Checking for any generated files:")
    !ls -R metrobot_gguf